# Day 3, Lesson 3.3: The Day 1 model as a tool

## Step 1: Load the model

In [1]:
import json
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from tools import live_weather
from tools.live_weather import get_live_weather, recent_hours
from tools.temperature_model import build_features, load_model, predict_from_window, predict_next_hour_temperature

saved = load_model()
print("Model:", type(saved["model"]).__name__)
print("Features (in order):", saved["features"])
print(f"Test error on 2024-2025 (ERA5): {saved['test_rmse']:.2f} °C")

Model: HistGradientBoostingRegressor
Features (in order): ['temp_now', 'temp_3h_ago', 'temp_avg_24h', 'sunlight', 'rain', 'hour']
Test error on 2024-2025 (ERA5): 0.41 °C


## Step 2: Build the 6 features from live data

In [2]:
last_24h = recent_hours(24)
print(f"Window: {last_24h.index[0]} -> {last_24h.index[-1]}  ({len(last_24h)} hours)")
build_features(last_24h)

Window: 2026-10-08 16:00:00 -> 2026-10-09 15:00:00  (24 hours)


,temp_now,temp_3h_ago,temp_avg_24h,sunlight,rain,hour
time,,,,,,
2026-10-09 15:00:00,35.4,35.8,32.025,521.0,0.0,15


## Step 3: Predict the next hour

In [3]:
prediction = predict_next_hour_temperature()
print(json.dumps(prediction, indent=2, ensure_ascii=False))

{
  "location": "Mumbai (South Mumbai)",
  "based_on_hour": "2026-10-09 15:00 IST",
  "forecast_for": "2026-10-09 16:00 IST",
  "last_observed_temperature_c": 35.4,
  "predicted_temperature_c": 34.7,
  "change_c": -0.7,
  "model": "Day 1 gradient boosting (6 features), trained on ERA5 1980-2023",
  "model_test_error_c": 0.41,
  "note": "Prediction from our own ML model; live inputs come from a different data source than training."
}


## Step 4: Honest check: our model vs Open-Meteo's forecast

In [4]:
live = get_live_weather()
target_hour = prediction["forecast_for"][11:16]
open_meteo = next((h["temperature_c"] for h in live["next_hours"] if h["time"] == target_hour), None)

print(f"Forecast for {prediction['forecast_for']}:")
print(f"  our Day 1 model : {prediction['predicted_temperature_c']} °C")
print(f"  Open-Meteo      : {open_meteo} °C")
print(f"  last observed   : {prediction['last_observed_temperature_c']} °C ({prediction['based_on_hour']})")

Forecast for 2026-10-09 16:00 IST:
  our Day 1 model : 34.7 °C
  Open-Meteo      : 34.2 °C
  last observed   : 35.4 °C (2026-10-09 15:00 IST)


## Step 5: Live backtest: how good is the model on LIVE data?

In [5]:
data = live_weather._fetch()
hourly = pd.DataFrame(data["hourly"])
hourly["time"] = pd.to_datetime(hourly["time"])
past = (hourly[hourly["time"] <= pd.Timestamp(data["current"]["time"])]
        .set_index("time")[["temperature_2m", "shortwave_radiation", "precipitation"]])

rows = []
for end in range(24, len(past)):
    window = past.iloc[end - 24:end]
    rows.append({"hour": past.index[end - 1].strftime("%d %H:%M"),
                 "model": predict_from_window(window),
                 "lazy_guess": window["temperature_2m"].iloc[-1],
                 "actual_next_hour": past["temperature_2m"].iloc[end]})
backtest = pd.DataFrame(rows)

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

print(f"Live backtest over {len(backtest)} hours:")
print(f"  model RMSE      {rmse(backtest['model'], backtest['actual_next_hour']):.2f} °C")
print(f"  lazy guess RMSE {rmse(backtest['lazy_guess'], backtest['actual_next_hour']):.2f} °C")
print(f"  (Day 1 test on ERA5: model {saved['test_rmse']:.2f} °C)")
backtest.tail(8)

Live backtest over 16 hours:
  model RMSE      0.69 °C
  lazy guess RMSE 0.99 °C
  (Day 1 test on ERA5: model 0.41 °C)


,hour,model,lazy_guess,actual_next_hour
8,09 07:00,29.5,28.7,30.1
9,09 08:00,30.4,30.1,32.5
10,09 09:00,34.0,32.5,34.7
11,09 10:00,35.5,34.7,35.7
12,09 11:00,35.9,35.7,35.8
13,09 12:00,36.1,35.8,35.5
14,09 13:00,35.2,35.5,36.0
15,09 14:00,34.6,36.0,35.4


## Step 6: The agent with two tools

In [6]:
import os
import time

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

from tools.live_weather import ToolError

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.5-flash-lite"

AGENT_RULES = """You are a weather assistant for operations staff in Mumbai.
- You only have data for Mumbai. For other cities, say you only cover Mumbai.
- For the temperature right now, or rain and conditions over the next hours: use get_live_weather.
- For the temperature in the NEXT HOUR: use predict_next_hour_temperature (our own trained model),
  and say it is our model's prediction.
- Use a tool only when it is needed. Never invent numbers. If a tool returns an error, say the data is unavailable."""

DECLARATIONS = [
    types.FunctionDeclaration(
        name="get_live_weather",
        description="LIVE Mumbai weather right now (temperature, feels-like, humidity, rain, wind, condition), "
                    "plus Open-Meteo's forecast for the next 6 hours and today's max/min.",
        parameters_json_schema={"type": "object", "properties": {}},
    ),
    types.FunctionDeclaration(
        name="predict_next_hour_temperature",
        description="Our own trained ML model (Day 1): predicts Mumbai's temperature for the NEXT HOUR "
                    "from the last 24 hours of live data.",
        parameters_json_schema={"type": "object", "properties": {}},
    ),
]
TOOL_FUNCTIONS = {"get_live_weather": get_live_weather,
                  "predict_next_hour_temperature": predict_next_hour_temperature}
CONFIG = types.GenerateContentConfig(
    system_instruction=AGENT_RULES,
    tools=[types.Tool(function_declarations=DECLARATIONS)],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    temperature=0.1,
)

def run_tool(name):
    try:
        return TOOL_FUNCTIONS[name]()
    except KeyError:
        return {"error": f"unknown tool {name}"}
    except ToolError as error:
        return {"error": str(error)}

def generate(contents, retries=4):
    for attempt in range(1, retries + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
        except errors.APIError as error:
            if error.code in (429, 503) and attempt < retries:
                wait = 30 if error.code == 429 else 5 * attempt
                print(f"   (Gemini {error.code}: waiting {wait}s)")
                time.sleep(wait)
                continue
            raise

def ask_agent(question, max_steps=4):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
    used = []
    for _ in range(max_steps):
        response = generate(contents)
        if not response.function_calls:
            return response.text.strip(), used
        contents.append(response.candidates[0].content)
        parts = []
        for call in response.function_calls:
            used.append(call.name)
            parts.append(types.Part.from_function_response(name=call.name, response=run_tool(call.name)))
        contents.append(types.Content(role="user", parts=parts))
    return "Sorry, I could not finish this question.", used

for question in [
    "How hot is it in Mumbai right now?",
    "What will the temperature be in Mumbai in one hour?",
    "Hello! Who are you?",
]:
    answer, used = ask_agent(question)
    print(f"Q: {question}\n   tools used: {used or 'none'}\n   answer: {answer}\n")

Q: How hot is it in Mumbai right now?
   tools used: ['get_live_weather']
   answer: In Mumbai right now (as of 15:00 IST), the temperature is **35.8°C** and it feels like **38.4°C** due to the humidity (36%). Conditions are mainly clear.



Q: What will the temperature be in Mumbai in one hour?
   tools used: ['predict_next_hour_temperature']
   answer: Based on our trained machine learning model, the predicted temperature in Mumbai for the next hour (at 16:00 IST) is **34.7°C** (a change of -0.7°C from the last observed temperature of 35.4°C).



Q: Hello! Who are you?
   tools used: none
   answer: Hello! I am your weather assistant for operations staff in Mumbai. I can help you with live weather updates, conditions, and temperature predictions for Mumbai using our custom forecasting model. How can I assist you today?

